# KrizKalkan AI · M3 — Türkçe Kriz Metin Motoru**Kaggle çalıştırma defteri.** Settings → Accelerator: **GPU T4 x2** (veya P100).Bu defter veriyi Kaggle'a yüklemez; `scripts/data/` betikleri veriyi HF'denyeniden üretir. Böylece defter ile depo arasında sürüm kayması olmaz.| Aşama | Veri | Öğrenilen | Tahmini süre (T4) ||---|---|---|---|| 1 | HumAID (52k, EN) | iddia tipi + yardım çağrısı | ~50 dk / 3 epok || 2 | MiDe22 + DMM (TR) | Türkçe yanlış bilgi | ~10 dk / 3 epok |⚠️ **Kota disiplini:** oturum bitince `Stop Session`. Her epok sonunda en iyiağırlık `/kaggle/working/` altına kaydedilir; oturum kesilse de kaybolmaz.

In [ ]:
# 1 · Depo ve bağımlılıklar!git clone -q https://github.com/devfurkank/KrizKalkanAI.git /kaggle/working/repo || echo "depo zaten var"%cd /kaggle/working/repo!git checkout -q feat/modeller && git pull -q   # kod güncellendiyse çeker!pip install -q "transformers>=4.46" "datasets>=3.1" "huggingface-hub>=0.26" 2>&1 | tail -2import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK ⚠️")

In [ ]:
# 2 · Veriyi üret (indir → doğrula → uyumlaştır → olay bazlı böl)!python scripts/data/fetch_text.py!python scripts/data/harmonize.py

## Aşama 1 — Kriz alanı (HumAID, İngilizce)Model kriz söyleminin yapısını, iddia tiplerini ve **yardım çağrısı** sınıfınıöğrenir. Türkçe etiketli yardım çağrısı verisi erişilebilir olmadığı için(bkz. `docs/veri-envanteri.md` · D3) bu sınıf çapraz dilli aktarımla öğrenilir;omurganın çok dilli olması bu yüzden zorunludur.

In [ ]:
# 3 · Aşama 1 — Kural 0 kayıp ağırlığı 8.0!python scripts/train/m3_text.py --asama 1 --epok 3 --yigin 32 --lr 3e-5 \    --yardim-agirlik 8.0 --cikti /kaggle/working/m3_asama1

## Aşama 2 — Türkçe uyarlama (MiDe22 + DMM)Aşama 1 ağırlıklarından devam eder. Gövde Türkçe kriz diline ve yanlış bilgietiketlerine uyarlanır; yardım çağrısı başlığı aşama 1'de öğrendiğini korur.

In [ ]:
# 4 · Aşama 2 — tekrar karışımı ile (unutmayı engeller)!python scripts/train/m3_text.py --asama 2 --epok 3 --yigin 32 --lr 2e-5 \    --replay 1.0 --yardim-agirlik 8.0 \    --devam /kaggle/working/m3_asama1 --cikti /kaggle/working/m3_asama2

In [ ]:
# 6 · Sonuçları özetle — bu tablo doğrudan model kartına girerimport json, pathlibfor d in sorted(pathlib.Path("/kaggle/working").glob("m3_asama*/egitim_raporu.json")):    r = json.loads(d.read_text())    print(f"\n═══ aşama {r['asama']} · eğitim {r['egitim_satir']:,} satır ═══")    for e in r["gecmis"]:        print("  ", json.dumps(e, ensure_ascii=False))

## Çıktıyı indirme`/kaggle/working/m3_asama2/` altındaki `model.pt` + tokenizer dosyalarını**Kaggle Dataset** olarak yayımlayın (Output → Create Dataset), sonra yerelde:```bashpython scripts/data/fetch_weights.py```Ağırlıklar `models/m3_text/` altına iner ve `KK_MODELS=on` ile devreye girer.Ağırlık inmezse sistem kural tabanlı yolla çalışmaya devam eder — demo çökmez.

## Kural 0 kontrolü — bu tablo geçmezse model kullanılamaz`yardim_duyarlilik_esikli` rapor hedefini (≥ 0,98) tutmalı **ve**`yardim_yanlis_pozitif_orani` makul kalmalı. Duyarlılık tek başına yanıltıcıdır:her içeriği "yardım çağrısı" sayan bir model de 1,00 duyarlılık verir amaKural 0'ı anlamsızlaştırır.Kabul ölçütü: duyarlılık ≥ 0,98 **ve** yanlış pozitif oranı ≤ 0,20.

In [ ]:
# 5 · Kural 0 kabul kontrolüimport json, pathlibfor d in sorted(pathlib.Path("/kaggle/working").glob("m3_asama*/egitim_raporu.json")):    r = json.loads(d.read_text())    son = r["gecmis"][-1]    print(f"\n═══ aşama {r['asama']} · eğitim {r['egitim_satir']:,} satır ═══")    for e in r["gecmis"]:        print("  ", json.dumps(e, ensure_ascii=False))    duy = son.get("yardim_duyarlilik_esikli")    fpr = son.get("yardim_yanlis_pozitif_orani")    if duy is not None:        tamam = duy >= 0.98 and (fpr is None or fpr <= 0.20)        print(f"\n  KURAL 0: duyarlılık {duy:.4f} · yanlış pozitif {fpr} "              f"→ {'✓ KABUL' if tamam else '🔴 RET — kayıp ağırlığını artırıp yeniden eğitin'}")

## Çıktıyı indirme`/kaggle/working/m3_asama2/` içeriğini Kaggle Dataset olarak yayımlayın.Model kartı **ölçüm yapılmadan yazılmaz** ve kart olmadan ağırlık yüklenmez(`models/registry.py` kabul kapısı). Yerelde:```bashKK_MODELS=on make model-durum```